<a href="https://colab.research.google.com/github/MarianoVIsabella/Data-Warehouse-Project/blob/main/DataCleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
BASE_PATH = '/content/drive/MyDrive'

In [3]:
!pip install scikit-learn --quiet

In [21]:
import pandas as pd
import numpy as np
import warnings
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
np.random.seed(42)
df_match_start = pd.read_csv(BASE_PATH + "/matches_1930_2022.csv")

# Data Cleaning Classes

In [5]:
class AuditLog:
    """
    Records every cleaning transformation applied to the data.
    Each entry captures: step name, column, row index, before, after, timestamp.
    """
    def __init__(self):
        self._entries = []

    def log(self, step: str, col: str, idx, before, after, reason: str = ''):

        self._entries.append({
            'step'      : step,
            'column'    : col,
            'row_index' : idx,
            'before'    : before,
            'after'     : after,
            'reason'    : reason,
            'timestamp' : datetime.now().isoformat()
        })

    def log_batch(self, step: str, col: str, mask: pd.Series,
                  before_series: pd.Series, after_series: pd.Series, reason: str = ''):
        changed_idx = mask[mask].index
        for idx in changed_idx:
            self.log(step, col, idx,
                     str(before_series.get(idx, 'N/A')),
                     str(after_series.get(idx, 'N/A')),
                     reason)

    def to_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._entries)

    def summary(self) -> pd.DataFrame:
        if not self._entries:
            return pd.DataFrame()
        df = self.to_df()
        return (df.groupby('step')
                  .agg(changes=('row_index','count'),
                       cols_affected=('column', lambda x: ', '.join(x.unique())))
                  .reset_index()
                  .sort_values('changes', ascending=False))

    def __len__(self):
        return len(self._entries)

In [20]:

class CleaningPipeline:

    def __init__(self, df: pd.DataFrame, table_name: str):
        self.original   = df.copy()
        self.df         = df.copy()
        self.table_name = table_name
        self.audit      = AuditLog()
        self._steps_run = []

    def standardize_strings(self, cols: list,
                              strip: bool = True,
                              lower: bool = False,
                              title_case: bool = False):

        for col in cols:
            if col not in self.df.columns: continue
            before = self.df[col].copy()
            s = self.df[col].astype(str)
            if strip:      s = s.str.strip()
            if lower:      s = s.str.lower()
            if title_case: s = s.str.title()
            changed = (s != before.astype(str)) & before.notna()
            self.df.loc[changed, col] = s[changed]
            self.audit.log_batch('standardize_strings', col, changed,
                                  before, self.df[col],
                                  'Strip whitespace + normalize case')

        self._steps_run.append('standardize_strings')
        return self

    def canonicalize_enum(self, col: str, mapping: dict,
                            unknown_value: str = 'Unknown'):

        if col not in self.df.columns: return self
        reverse = {}
        for canonical, variants in mapping.items():
            for v in variants:
                reverse[v.strip().lower()] = canonical
        before = self.df[col].copy()

        def _map(val):
            if pd.isna(val): return val
            return reverse.get(str(val).strip().lower(), unknown_value)

        self.df[col] = self.df[col].apply(_map)
        changed = (self.df[col] != before) & before.notna()
        self.audit.log_batch('canonicalize_enum', col, changed,
                              before, self.df[col],
                              f'Canonical mapping for {col}')
        self._steps_run.append(f'canonicalize_enum:{col}')

        return self

    def impute(self, col: str, strategy: str = 'median',
                group_col: str = None, mnar_flag: bool = False):
        """
        strategy: 'mean' | 'median' | 'mode' | 'constant:VALUE' | 'mnar_flag'
        group_col: if set, compute statistic within each group (MAR strategy)
        mnar_flag: if True, add a binary indicator column instead of imputing
        """
        if col not in self.df.columns: return self
        miss_mask = self.df[col].isna()
        if not miss_mask.any(): return self

        if mnar_flag:
            flag_col = f'{col}_missing_flag'
            self.df[flag_col] = miss_mask.astype(int)
            self.audit.log('impute', col, 'ALL', 'null', f'flag → {flag_col}', 'MNAR — do not impute')
            self._steps_run.append(f'mnar_flag:{col}')

            return self

        before = self.df[col].copy()
        if group_col and group_col in self.df.columns:

            if strategy == 'median':
                fill_vals = self.df.groupby(group_col)[col].transform('median')
            elif strategy == 'mean':
                fill_vals = self.df.groupby(group_col)[col].transform('mean')
            else:
                fill_vals = self.df.groupby(group_col)[col].transform(
                    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
            self.df[col] = self.df[col].fillna(fill_vals)

        elif strategy.startswith('constant:'):
            val = strategy.split(':')[1]
            self.df[col] = self.df[col].fillna(val)

        elif strategy in ('mean', 'median'):
            stat_val = getattr(self.df[col], strategy)()
            self.df[col] = self.df[col].fillna(stat_val)

        elif strategy == 'mode':
            self.df[col] = self.df[col].fillna(self.df[col].mode().iloc[0])

        changed = miss_mask & self.df[col].notna()
        self.audit.log_batch('impute', col, changed,
                              before, self.df[col],
                              f'strategy={strategy}, group={group_col}')
        self._steps_run.append(f'impute:{col}')

        return self

    def normalize(self, cols: list, method: str = 'minmax'):
        valid_cols = [c for c in cols if c in self.df.columns
                      and pd.api.types.is_numeric_dtype(self.df[c])]
        X = self.df[valid_cols].fillna(self.df[valid_cols].median())

        if method == 'minmax':
            scaler = MinMaxScaler()
            result = scaler.fit_transform(X)

        elif method == 'zscore':
            scaler = StandardScaler()
            result = scaler.fit_transform(X)

        elif method == 'log1p':
            result = np.log1p(np.clip(X, 0, None))

        else:
            raise ValueError(f'Unknown method: {method}')

        for i, col in enumerate(valid_cols):
            new_col = f'{col}_{method}'
            self.df[new_col] = result[:, i]
            self.audit.log('normalize', col, 'ALL', 'raw', new_col,
                           f'method={method} → new col {new_col}')
        self._steps_run.append(f'normalize:{method}')

        return self

    def summary(self):
        orig_rows = len(self.original)
        curr_rows = len(self.df)
        print(f'\n{"═"*55}')
        print(f'  CLEANING PIPELINE SUMMARY — {self.table_name}')
        print(f'{"═"*55}')
        print(f'  Original rows  : {orig_rows}')
        print(f'  Current rows   : {curr_rows} ({orig_rows - curr_rows} removed)')
        print(f'  Steps run      : {len(self._steps_run)}')
        print(f'  Audit entries  : {len(self.audit)}')
        print(f'{"─"*55}')
        print(self.audit.summary().to_string(index=False))
        print(f'{"═"*55}')

    @property
    def clean_df(self) -> pd.DataFrame:
        return self.df.copy()


# Data Cleaning tasks


1.   The very first thing we do is to ensure all the text fields in the dataset doesn't begin or end with spaces. This step will solve the validity issue on the Score column.



In [22]:
textual_fields=['Score','home_team','away_team','home_manager','away_manager','home_captain','away_captain','Venue',
                'Officials','Round','Referee','Host','Notes','home_goal','away_goal','home_goal_long','away_goal_long',
                'home_own_goal','away_own_goal','home_penalty','away_penalty','home_penalty_goal','away_penalty_goal',
                'home_penalty_miss_long','away_penalty_miss_long','home_penalty_shootout_goal_long','away_penalty_shootout_goal_long',
                'home_penalty_shootout_miss_long','away_penalty_shootout_miss_long','home_red_card','away_red_card',
                'home_yellow_red_card','away_yellow_red_card','home_yellow_card_long','away_yellow_card_long','home_substitute_in_long',
                'away_substitute_in_long']
nations_map = {
    'DR. Congo' : ['Zaire'],
}
pipe_prod = (
    CleaningPipeline(df_match_start, 'match_fact')
    .standardize_strings(textual_fields, strip=True, title_case=False)
    .canonicalize_enum('home_team', nations_map)
    .canonicalize_enum('away_team', nations_map)
)

pipe_prod.summary()
clean_products_output_path = BASE_PATH + '/matches_clean.csv'
products_audit_output_path = BASE_PATH + '/matches_audit_log.csv'
products_audit_summary_output_path = BASE_PATH + '/matches_audit_summary.csv'
pipe_prod.clean_df.to_csv(clean_products_output_path, index=False)
pipe_prod.audit.to_df().to_csv(products_audit_output_path, index=False)
pipe_prod.audit.summary().to_csv(products_audit_summary_output_path, index=False)


═══════════════════════════════════════════════════════
  CLEANING PIPELINE SUMMARY — match_fact
═══════════════════════════════════════════════════════
  Original rows  : 964
  Current rows   : 964 (0 removed)
  Steps run      : 3
  Audit entries  : 1929
───────────────────────────────────────────────────────
               step  changes        cols_affected
  canonicalize_enum     1928 home_team, away_team
standardize_strings        1                Score
═══════════════════════════════════════════════════════


Task da fare:
Gestire nomi nazioni diversi nel tempo
Gestire nomi round varie edizioni
Mantieni home_penalty e Away_penalty interi?

In [17]:
nations=set()
"""
issues:
Zaire -> Former Name of DR Congo
Dutch East Indies -> Colonial Name of Indonesia
Czechoslovakia -> Czech Republic and Slovakia
Germany DR -> Germany?
West Germany -> Germany?
Serbia and Montenegro -> Serbia + Montenegro ?
Yugoslavia -> All former nations composing Yugoslavia
Soviet Union -> Russia (other States too?)
"""
for row in df_match_start["home_team"]:
  nations.add(row)
for row in df_match_start["away_team"]:
  nations.add(row)

nations

{'Algeria',
 'Angola',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Brazil',
 'Bulgaria',
 'Cameroon',
 'Canada',
 'Chile',
 'China PR',
 'Colombia',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Czechoslovakia',
 "Côte d'Ivoire",
 'Denmark',
 'Dutch East Indies',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'FR Yugoslavia',
 'France',
 'Germany',
 'Germany DR',
 'Ghana',
 'Greece',
 'Haiti',
 'Honduras',
 'Hungary',
 'IR Iran',
 'Iceland',
 'Iraq',
 'Israel',
 'Italy',
 'Jamaica',
 'Japan',
 'Korea DPR',
 'Korea Republic',
 'Kuwait',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Nigeria',
 'Northern Ireland',
 'Norway',
 'Panama',
 'Paraguay',
 'Peru',
 'Poland',
 'Portugal',
 'Qatar',
 'Republic of Ireland',
 'Romania',
 'Russia',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'Serbia',
 'Serbia and Montenegro',
 'Slovakia',
 'Slovenia',
 'South Africa',
 'Soviet Union',
 'Spain',
 'Sweden',
 'Switzerland',
 'Tog

In [18]:
rounds = set()
#Former phases not anymore standing: Final Stage, First group stage, First round, Group stage play-off, Second group stage,
#Second Round
for row in df_match_start["Round"]:
    rounds.add(row)
rounds

{'Final',
 'Final stage',
 'First group stage',
 'First round',
 'Group stage',
 'Group stage play-off',
 'Quarter-finals',
 'Round of 16',
 'Second group stage',
 'Second round',
 'Semi-finals',
 'Third-place match'}